# 🧠 Phase 2: Semantic Feature Extraction (SAM 2 + CLIP)

This notebook extracts 2D semantic supervisory signals from D-NeRF training frames using:
- **SAM 2** (Segment Anything 2) for automatic object mask generation
- **CLIP** (ViT-B/32) for per-mask semantic feature extraction and text-guided labeling

**Outputs** (saved to `data/dnerf/<scene>/semantics/`):
- Per-frame binary masks with metadata (`masks/`)
- Per-pixel integer label maps (`labels/`)
- Per-pixel dense CLIP feature maps (`features/`)
- Text prompt embeddings and pipeline metadata

These outputs serve as ground-truth supervision for Phase 3 (semantic lifting into 4D Gaussians).

> **Prerequisites**: Phase 1 must be completed first (dataset must be downloaded).
> **Runtime**: ~10-20 minutes on T4 GPU depending on scene size.

In [ ]:
# Cell 1: Environment Setup
import torch
import os

# GPU check
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA: {torch.version.cuda}")
else:
    print("⚠️ No GPU detected. SAM 2 will run slowly on CPU.")
    print("   Go to Runtime → Change runtime type → GPU")

# Navigate to project directory
if os.path.exists("/content/4DGaussians"):
    %cd /content/4DGaussians
    print(f"\n✅ Working directory: {os.getcwd()}")
else:
    print("❌ 4DGaussians not found. Run Phase 1 notebook first!")
    print("   Or clone: !git clone https://github.com/hustvl/4DGaussians.git /content/4DGaussians")

In [ ]:
# Cell 2: Install SAM 2, CLIP, and other dependencies
print("Installing Phase 2 dependencies...\n")

# SAM 2 - Meta's Segment Anything 2
!pip install -q segment-anything-2 2>/dev/null || \
    pip install -q git+https://github.com/facebookresearch/sam2.git

# OpenCLIP for CLIP feature extraction
!pip install -q open-clip-torch

# Other utilities
!pip install -q opencv-python matplotlib imageio[ffmpeg] tqdm

print("\n✅ All Phase 2 dependencies installed!")

# Verify imports
try:
    from sam2.build_sam import build_sam2
    from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
    print("  ✓ SAM 2 imported successfully")
except ImportError as e:
    print(f"  ⚠️ SAM 2 import failed: {e}")
    print("  Trying alternative installation...")
    !pip install -q segment-anything-2

try:
    import open_clip
    print("  ✓ OpenCLIP imported successfully")
except ImportError as e:
    print(f"  ⚠️ OpenCLIP import failed: {e}")

## 📥 Download SAM 2 Checkpoint
Downloading the SAM 2 Hiera-Small model checkpoint (~150 MB).

In [ ]:
# Cell 3: Download SAM 2 model weights
import os

checkpoint_dir = "/content/4DGaussians/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

sam2_checkpoint = os.path.join(checkpoint_dir, "sam2_hiera_small.pt")

if not os.path.exists(sam2_checkpoint):
    print("Downloading SAM 2 Hiera-Small checkpoint (~150 MB)...")
    !wget -q --show-progress \
        https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt \
        -O {sam2_checkpoint}
    print(f"\n✅ SAM 2 checkpoint saved: {sam2_checkpoint}")
else:
    print(f"SAM 2 checkpoint already exists: {sam2_checkpoint}")

# Verify file size
size_mb = os.path.getsize(sam2_checkpoint) / (1024 * 1024)
print(f"   Size: {size_mb:.1f} MB")

## ⚙️ Configure Extraction Pipeline
Set up the scene, text prompts, and extraction parameters.

In [ ]:
# Cell 4: Configure the semantic extraction pipeline
import sys
sys.path.insert(0, "/content/4DGaussians")

from preprocess.config import SemanticConfig

# ── Configure ────────────────────────────────────────────────
SCENE = "bouncingballs"  # Change this for different scenes
DATASET_PATH = f"/content/4DGaussians/data/dnerf/{SCENE}"
SAM2_CHECKPOINT = "/content/4DGaussians/checkpoints/sam2_hiera_small.pt"

# Text prompts for semantic labeling
# Customize these based on the objects in your scene:
TEXT_PROMPTS = [
    "background",     # Class 0 (after +1 offset): catch-all
    "ball",           # Class 1: the bouncing balls  
    "floor",          # Class 2: the ground plane
    "wall",           # Class 3: walls/boundaries
]

config = SemanticConfig(
    dataset_path=DATASET_PATH,
    sam2_checkpoint=SAM2_CHECKPOINT,
    sam2_model_cfg="sam2_hiera_s",
    clip_model_name="ViT-B-32",
    clip_pretrained="openai",
    clip_feature_dim=512,
    text_prompts=TEXT_PROMPTS,
    sam_points_per_side=32,        # 32x32 grid = 1024 prompt points per image
    sam_pred_iou_thresh=0.7,
    sam_stability_score_thresh=0.85,
    max_masks_per_frame=64,
    batch_size=8,                  # CLIP batch size (increase if GPU memory allows)
    device="cuda" if torch.cuda.is_available() else "cpu",
    save_float16=True,             # Save features as float16 to save disk space
    process_test_frames=True,
    verbose=True,
)

print("Configuration:")
print(f"  Scene: {SCENE}")
print(f"  Dataset: {config.dataset_path}")
print(f"  Output: {config.output_dir}")
print(f"  SAM 2: {config.sam2_model_cfg}")
print(f"  CLIP: {config.clip_model_name}")
print(f"  Text prompts: {config.text_prompts}")
print(f"  Device: {config.device}")

## 🔬 Run Semantic Extraction
Processing all frames through the SAM 2 + CLIP pipeline.
For each frame:
1. SAM 2 generates object-level masks automatically
2. CLIP encodes each masked region into a 512-dim feature vector
3. Text-prompt cosine similarity assigns semantic class labels
4. Per-pixel label maps and dense feature maps are saved

In [ ]:
# Cell 5: Run the full semantic extraction pipeline
from preprocess.extract_semantics import SemanticExtractor

extractor = SemanticExtractor(config)
extractor.run()

## 🎨 Visualize Results
Inspecting the extracted semantic masks, label maps, and CLIP similarity heatmaps.

In [ ]:
# Cell 6: Visualize extraction results for a sample frame
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from preprocess.visualize_semantics import (
    load_metadata, load_label_map, load_masks,
    colorize_label_map, overlay_masks_on_image, 
    visualize_frame, visualize_clip_similarity
)

semantics_dir = config.output_dir

# Load metadata
metadata = load_metadata(semantics_dir)
print("Pipeline Metadata:")
print(json.dumps(metadata["statistics"], indent=2))
print(f"\nClass mapping: {metadata['class_mapping']}")

# Visualize a sample frame
FRAME_NAME = "r_0"  # Change to visualize different frames

# Find the image path
with open(f"{config.dataset_path}/transforms_train.json") as f:
    train_data = json.load(f)

frame_path = None
for frame in train_data["frames"]:
    from pathlib import Path
    if Path(frame["file_path"]).stem == FRAME_NAME:
        fp = Path(config.dataset_path) / frame["file_path"]
        if not fp.suffix:
            fp = fp.with_suffix(".png")
        frame_path = str(fp)
        break

if frame_path:
    visualize_frame(
        image_path=frame_path,
        semantics_dir=semantics_dir,
        frame_name=FRAME_NAME,
        save_path=f"{semantics_dir}/visualizations/{FRAME_NAME}_viz.png",
        show=True,
    )

In [ ]:
# Cell 7: CLIP text similarity heatmaps
from preprocess.visualize_semantics import visualize_clip_similarity
import os

os.makedirs(f"{semantics_dir}/visualizations", exist_ok=True)

visualize_clip_similarity(
    semantics_dir=semantics_dir,
    frame_name=FRAME_NAME,
    save_path=f"{semantics_dir}/visualizations/{FRAME_NAME}_clip_sim.png",
    show=True,
)

print("\nEach heatmap shows per-pixel cosine similarity to the text prompt.")
print("Brighter regions = higher similarity to that semantic category.")

In [ ]:
# Cell 8: Generate semantic overlay video
from preprocess.visualize_semantics import create_semantic_video

video_output = f"{semantics_dir}/visualizations/semantic_overlay.mp4"

create_semantic_video(
    dataset_path=config.dataset_path,
    semantics_dir=semantics_dir,
    output_path=video_output,
    fps=15,
    split="train",
)

# Display video inline
from IPython.display import HTML
from base64 import b64encode
import os

if os.path.exists(video_output):
    with open(video_output, "rb") as f:
        video_data = f.read()
    b64_video = b64encode(video_data).decode()
    html = f'''
    <video width="800" controls autoplay loop>
        <source src="data:video/mp4;base64,{b64_video}" type="video/mp4">
    </video>
    <p style="text-align:center; color:gray">
        Left: Original | Right: Semantic overlay (SAM 2 + CLIP labels)
    </p>
    '''
    display(HTML(html))

## ✅ Verify Output Structure
Confirm all expected files were generated correctly.

In [ ]:
# Cell 9: Verify output completeness and integrity
import os
import json
import numpy as np
from pathlib import Path

semantics_dir = config.output_dir

print("=" * 60)
print("  Phase 2 Output Verification")
print("=" * 60)

# Check metadata
metadata_path = os.path.join(semantics_dir, "metadata.json")
assert os.path.exists(metadata_path), "❌ metadata.json missing!"
with open(metadata_path) as f:
    metadata = json.load(f)
print(f"\n✓ metadata.json — {metadata['statistics']['total_frames_processed']} frames processed")

# Check text features
text_feat_path = os.path.join(semantics_dir, "clip_text_features.npy")
assert os.path.exists(text_feat_path), "❌ clip_text_features.npy missing!"
text_feats = np.load(text_feat_path)
expected_shape = (len(config.text_prompts), config.clip_feature_dim)
assert text_feats.shape == expected_shape, f"❌ Shape mismatch: {text_feats.shape} != {expected_shape}"
print(f"✓ clip_text_features.npy — shape {text_feats.shape}")

# Check masks, labels, and features for each frame
masks_dir = os.path.join(semantics_dir, "masks")
labels_dir = os.path.join(semantics_dir, "labels")
features_dir = os.path.join(semantics_dir, "features")

mask_files = sorted(Path(masks_dir).glob("*_masks.npz"))
label_files = sorted(Path(labels_dir).glob("*_labels.png"))
feature_files = sorted(Path(features_dir).glob("*_features.npy"))

print(f"\n✓ masks/ — {len(mask_files)} files")
print(f"✓ labels/ — {len(label_files)} files")
print(f"✓ features/ — {len(feature_files)} files")

# Spot-check a random feature file
if feature_files:
    sample_feat = np.load(str(feature_files[0]))
    H, W = config.image_size, config.image_size
    print(f"\n  Sample feature map shape: {sample_feat.shape}")
    print(f"  dtype: {sample_feat.dtype}")
    assert sample_feat.shape[2] == config.clip_feature_dim, \
        f"❌ Feature dim mismatch: {sample_feat.shape[2]} != {config.clip_feature_dim}"
    print(f"  ✓ Feature dimension matches CLIP ({config.clip_feature_dim})")

# Spot-check a label map
if label_files:
    from PIL import Image
    sample_label = np.array(Image.open(str(label_files[0])))
    unique_labels = np.unique(sample_label)
    max_valid_label = len(config.text_prompts) + 1
    print(f"\n  Sample label map shape: {sample_label.shape}")
    print(f"  Unique labels: {unique_labels.tolist()}")
    assert all(l <= max_valid_label for l in unique_labels), \
        f"❌ Invalid label IDs found: {unique_labels}"
    print(f"  ✓ All label IDs valid (0..{max_valid_label})")

# Calculate total disk usage
total_size = 0
for dirpath, _, filenames in os.walk(semantics_dir):
    for f in filenames:
        total_size += os.path.getsize(os.path.join(dirpath, f))

print(f"\n  Total disk usage: {total_size / (1024*1024):.1f} MB")

print("\n" + "=" * 60)
print("  ✅ Phase 2 Complete — Semantic Features Extracted!")
print("=" * 60)
print("\n  Output directory structure:")
print(f"  {semantics_dir}/")
print(f"  ├── metadata.json")
print(f"  ├── clip_text_features.npy")
print(f"  ├── masks/  ({len(mask_files)} files)")
print(f"  ├── labels/ ({len(label_files)} files)")
print(f"  ├── features/ ({len(feature_files)} files)")
print(f"  └── visualizations/")
print(f"\n  Next: Run Phase 3 to lift these features into 4D Gaussians!")